# 第四章：向量存储与持久化

## 学习目标
- 理解 LlamaIndex 的存储架构（StorageContext）
- 配置 FAISS 作为向量存储后端
- 控制 Embedding 模型配置
- 实现索引的持久化（保存和加载）
- 对比 LangChain FAISS 的使用方式

## 0. 环境初始化

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv("../.env")

from llama_index.llms.openai_like import OpenAILike
from llama_index.embeddings.openai_like import OpenAILikeEmbedding
from llama_index.core import Settings

# 配置全局 LLM
Settings.llm = OpenAILike(
    model="qwen-plus",
    api_base=os.environ["Qwen_API_BASE"],
    api_key=os.environ["Qwen_API_KEY"],
    is_chat_model=True,
)

# 切换为 GLM（同样是 OpenAI 兼容接口）：
# from llama_index.llms.openai_like import OpenAILike
# Settings.llm = OpenAILike(model="glm-4-plus", api_base=os.environ["GLM_API_BASE"], api_key=os.environ["GLM_API_KEY"], is_chat_model=True)

# 配置全局 Embedding 模型
Settings.embed_model = OpenAILikeEmbedding(
    model_name="text-embedding-v3",
    api_base=os.environ["Qwen_API_BASE"],
    api_key=os.environ["Qwen_API_KEY"],
)

print(f"LLM: {Settings.llm.model}")
print(f"Embedding: {Settings.embed_model.model_name}")

## 1. 默认的内存向量存储

前几章我们用 `VectorStoreIndex.from_documents()` 构建索引时，没有指定任何存储后端。这时 LlamaIndex 默认使用一个**内存中的简单向量存储**（`SimpleVectorStore`）。

先准备一组关于 AI 主题的文档：

In [ ]:
from llama_index.core import Document, VectorStoreIndex

documents = [
    Document(
        text="机器学习是人工智能的一个子领域，它使计算机能够从数据中学习而无需显式编程。监督学习、无监督学习和强化学习是三种主要的学习范式。",
        metadata={"topic": "机器学习"},
    ),
    Document(
        text="深度学习是机器学习的一个分支，使用多层神经网络来学习数据的层次化表示。CNN 擅长图像处理，RNN 和 Transformer 擅长序列数据处理。",
        metadata={"topic": "深度学习"},
    ),
    Document(
        text="自然语言处理（NLP）是 AI 处理人类语言的技术。现代 NLP 主要基于 Transformer 架构，如 BERT 用于理解，GPT 用于生成。",
        metadata={"topic": "NLP"},
    ),
    Document(
        text="计算机视觉是让机器理解图像和视频的 AI 技术。卷积神经网络（CNN）是其核心技术，应用包括图像分类、目标检测和语义分割。",
        metadata={"topic": "计算机视觉"},
    ),
    Document(
        text="强化学习是一种通过与环境交互来学习最优策略的方法。Agent 在环境中采取行动获取奖励，目标是最大化长期累积奖励。AlphaGo 是强化学习的著名应用。",
        metadata={"topic": "强化学习"},
    ),
]

print(f"准备了 {len(documents)} 个文档")

In [ ]:
# 默认方式构建索引（内存存储）
index_default = VectorStoreIndex.from_documents(documents)
print(f"索引类型: {type(index_default).__name__}")
print(f"存储后端: {type(index_default._vector_store).__name__}")

### 刚才发生了什么？

`VectorStoreIndex.from_documents()` 在不指定存储后端时，会自动创建一个 `SimpleVectorStore`——这是一个纯 Python 实现的内存向量存储，把所有向量存在一个字典里。

**问题在哪？**
- 文档量大时，内存占用高、检索速度慢
- 程序重启后，所有向量数据丢失
- 不支持高效的近似最近邻搜索（ANN）

对于生产环境，我们需要专业的向量存储后端。FAISS 就是最常用的选择之一——你在 LangChain 第五章已经用过它了。

## 2. 使用 FAISS 向量存储

FAISS（Facebook AI Similarity Search）是 Meta 开发的高效向量相似度搜索库。在 LlamaIndex 中使用 FAISS 需要通过 `StorageContext` 这个核心概念。

In [ ]:
# 先检测当前 Embedding 模型的向量维度
test_embedding = Settings.embed_model.get_text_embedding("测试文本")
d = len(test_embedding)
print(f"Embedding 模型输出维度: {d}")

### 常见问题

> **为什么要先检测维度？**
>
> FAISS 索引在创建时需要指定向量维度，这个维度必须和 Embedding 模型的输出维度**完全一致**。
> DashScope 的 `text-embedding-v3` 默认维度是 1024，但不同模型或不同配置可能不同。
> 用 `get_text_embedding()` 实际测试一次最保险。
>
> 如果后续遇到维度不匹配的报错，检查这个值是否正确。

In [ ]:
import faiss
from llama_index.vector_stores.faiss import FaissVectorStore
from llama_index.core import StorageContext

# 创建 FAISS 索引（L2 距离，精确搜索）
faiss_index = faiss.IndexFlatL2(d)

# 用 FAISS 创建向量存储
vector_store = FaissVectorStore(faiss_index=faiss_index)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

# 构建索引（使用 FAISS 存储）
index = VectorStoreIndex.from_documents(
    documents,
    storage_context=storage_context,
)
print("✓ 使用 FAISS 构建索引完成")
print(f"FAISS 索引中的向量数: {faiss_index.ntotal}")

### 刚才发生了什么？

这段代码做了三件事：

1. **创建 FAISS 索引**：`faiss.IndexFlatL2(d)` 创建一个基于 L2 距离（欧几里得距离）的精确搜索索引。参数 `d` 是向量维度。

2. **封装为 LlamaIndex 向量存储**：`FaissVectorStore(faiss_index=faiss_index)` 把原生 FAISS 索引包装成 LlamaIndex 能识别的 `VectorStore` 接口。

3. **通过 StorageContext 注入**：`StorageContext.from_defaults(vector_store=vector_store)` 创建存储上下文，再传给 `from_documents()`。

### StorageContext 是什么？

`StorageContext` 是 LlamaIndex 的**存储统一管理层**，它封装了三种存储：

| 存储组件 | 存储内容 | 默认实现 |
|---------|---------|--------|
| **vector_store** | 向量 Embedding | `SimpleVectorStore`（内存） |
| **docstore** | 原始文档/节点的文本和元数据 | `SimpleDocumentStore`（内存） |
| **index_store** | 索引结构信息 | `SimpleIndexStore`（内存） |

通过替换其中任意一个组件，就能切换存储后端，而上层的索引构建和查询代码**完全不需要改动**。这就是 StorageContext 的价值。

## 3. 查询 FAISS 索引

切换到 FAISS 后端后，查询接口和之前完全一样——这就是 StorageContext 抽象的好处。

In [ ]:
# 查询（和之前完全一样的接口）
query_engine = index.as_query_engine(similarity_top_k=2)
response = query_engine.query("深度学习和机器学习是什么关系？")
print(response)
print()
for node in response.source_nodes:
    print(f"  [{node.metadata['topic']}] 相似度: {node.score:.4f}")

### 刚才发生了什么？

- `similarity_top_k=2` 指定返回最相似的 2 个文档节点
- `response.source_nodes` 包含了检索到的源文档，每个节点有 `score`（相似度分数）和 `metadata`
- 注意 FAISS 使用 L2 距离，所以 score 越**小**越相似（与余弦相似度的越大越相似相反）

查询接口没有任何变化——无论底层是 `SimpleVectorStore` 还是 FAISS，上层代码都是 `as_query_engine().query()`。

## 4. 持久化：保存索引

内存中的索引在程序退出后就丢失了。LlamaIndex 通过 `StorageContext.persist()` 将所有存储组件一次性保存到磁盘。

In [ ]:
# 保存索引到磁盘
PERSIST_DIR = "./storage_demo"
storage_context.persist(persist_dir=PERSIST_DIR)
print(f"✓ 索引已保存到 {PERSIST_DIR}/")

In [ ]:
# 查看保存了什么文件
import os
for f in sorted(os.listdir(PERSIST_DIR)):
    size = os.path.getsize(os.path.join(PERSIST_DIR, f))
    print(f"  {f} ({size:,} bytes)")

### 刚才发生了什么？

`persist()` 将三种存储分别保存为文件：

| 文件 | 对应存储 | 内容 |
|------|---------|------|
| `default__vector_store.faiss` | vector_store | FAISS 二进制索引文件，包含所有向量 |
| `docstore.json` | docstore | 文档/节点的原始文本和元数据（JSON 格式） |
| `index_store.json` | index_store | 索引结构元信息（JSON 格式） |

> **注意**：FAISS 的持久化文件是二进制格式（不是 JSON），这和 `SimpleVectorStore` 的纯 JSON 存储不同。FAISS 的二进制格式更紧凑、加载更快。

## 5. 从磁盘加载索引

模拟程序重启后的场景：从磁盘重新加载索引，无需重新计算 Embedding。

In [ ]:
from llama_index.core import load_index_from_storage

# 重新创建存储上下文（从磁盘）
vector_store = FaissVectorStore.from_persist_dir(PERSIST_DIR)
storage_context = StorageContext.from_defaults(
    vector_store=vector_store,
    persist_dir=PERSIST_DIR,
)

# 加载索引
loaded_index = load_index_from_storage(storage_context)
print("✓ 索引加载成功")

In [ ]:
# 验证：查询加载后的索引
response = loaded_index.as_query_engine().query("什么是强化学习？")
print(response)
print()
for node in response.source_nodes:
    print(f"  [{node.metadata['topic']}] 相似度: {node.score:.4f}")

### 刚才发生了什么？

加载分两步：

1. **恢复存储上下文**：
   - `FaissVectorStore.from_persist_dir()` 从磁盘加载 FAISS 二进制索引
   - `StorageContext.from_defaults(persist_dir=...)` 从磁盘加载 docstore 和 index_store

2. **恢复索引**：
   - `load_index_from_storage()` 根据 index_store 中的元信息，重建完整的 `VectorStoreIndex` 对象

**关键优势**：加载时不需要重新调用 Embedding API——向量已经存在 FAISS 文件中了。对于大量文档，这能节省大量时间和 API 费用。

> **注意**：FAISS 的加载方式和 SimpleVectorStore 不同。SimpleVectorStore 只需要 `StorageContext.from_defaults(persist_dir=...)` 就够了，但 FAISS 需要先用 `FaissVectorStore.from_persist_dir()` 单独加载向量存储。这是因为 FAISS 使用自己的二进制格式，不能用通用的 JSON 加载。

## 6. 对比：LangChain vs LlamaIndex 的 FAISS 使用

你在 LangChain 第五章已经学过 FAISS 的使用。下面对比两个框架的差异：

| 操作 | LangChain | LlamaIndex |
|------|-----------|------------|
| **构建** | `FAISS.from_documents(docs, embeddings)` | `VectorStoreIndex.from_documents(docs, storage_context=...)` |
| **保存** | `vectorstore.save_local(path)` | `storage_context.persist(persist_dir=path)` |
| **加载** | `FAISS.load_local(path, embeddings)` | `load_index_from_storage(StorageContext.from_defaults(...))` |
| **检索** | `vectorstore.as_retriever()` | `index.as_retriever()` |
| **查询+生成** | 需手动构建 LCEL 链 | `index.as_query_engine()` |

### 关键差异

**LangChain 的方式更直接**：
- FAISS 就是向量存储本身，直接操作它
- 保存/加载是向量存储自己的方法
- 但要实现 RAG（检索+生成），需要自己用 LCEL 链把检索器、提示词、LLM 串起来

**LlamaIndex 的方式更抽象**：
- FAISS 被包装在 `StorageContext` 里，通过统一接口操作
- 持久化由 StorageContext 统一管理（一次保存三种存储）
- `as_query_engine()` 一步到位，内置了检索+生成的完整流程

> **一句话总结**：LangChain 给你积木让你搭，LlamaIndex 给你成品让你用。各有适合的场景——需要高度定制时选 LangChain，快速搭建 RAG 时选 LlamaIndex。

## 7. 向已有索引添加文档

实际应用中，知识库不是一次性构建的，需要持续添加新文档。LlamaIndex 支持用 `insert()` 向已有索引中追加文档。

In [ ]:
# 新增文档
new_doc = Document(
    text="迁移学习是将一个领域学到的知识应用到另一个领域的技术。预训练模型如 BERT、GPT 就是迁移学习的典型应用，先在大规模数据上预训练，再在特定任务上微调。",
    metadata={"topic": "迁移学习"},
)

# 插入到已有索引
loaded_index.insert(new_doc)
print("✓ 新文档已插入索引")

In [ ]:
# 验证：查询新插入的内容
response = loaded_index.as_query_engine().query("什么是迁移学习？")
print(response)
print()
for node in response.source_nodes:
    print(f"  [{node.metadata['topic']}] 相似度: {node.score:.4f}")

### 刚才发生了什么？

`insert()` 会：
1. 将文档切分为节点（使用全局配置的文本切分器）
2. 调用 Embedding 模型计算向量
3. 将向量插入 FAISS 索引
4. 将文档文本和元数据存入 docstore

**与 LangChain 的对比**：
- LangChain FAISS 用 `vectorstore.add_documents([doc])` 添加文档
- LlamaIndex 用 `index.insert(doc)` 添加文档
- 两者都支持增量添加，但 LlamaIndex 的 `insert()` 会自动处理切分和索引更新

> **注意**：`insert()` 后记得再次 `persist()` 保存，否则新增的文档只在内存中，重启后丢失。

## 8. Embedding 模型配置

向量存储的质量直接取决于 Embedding 模型。了解如何检查和配置 Embedding 模型很重要。

In [ ]:
# 查看当前 Embedding 模型
print(f"模型: {Settings.embed_model.model_name}")

# 测试 Embedding
test_embedding = Settings.embed_model.get_text_embedding("测试文本")
print(f"向量维度: {len(test_embedding)}")
print(f"前 5 个值: {test_embedding[:5]}")

### Embedding 配置要点

| 配置项 | 说明 |
|--------|------|
| `Settings.embed_model` | 全局 Embedding 模型，所有索引默认使用 |
| `model_name` | 模型名称，决定输出维度和效果 |
| `get_text_embedding()` | 获取单个文本的向量，可用于测试 |
| `get_text_embedding_batch()` | 批量获取向量，效率更高 |

**维度匹配原则**：
- FAISS 索引创建时的维度 `d` 必须等于 Embedding 模型的输出维度
- 如果切换了 Embedding 模型，需要重新创建 FAISS 索引并重新构建所有向量
- 不能混用不同维度的 Embedding 模型——旧向量和新向量不兼容

> **实用技巧**：在构建 FAISS 索引前，始终用 `len(Settings.embed_model.get_text_embedding("test"))` 检测实际维度，避免硬编码。

## 9. 清理

删除本章创建的临时存储目录。

In [ ]:
import shutil
if os.path.exists(PERSIST_DIR):
    shutil.rmtree(PERSIST_DIR)
    print("✓ 已清理存储目录")

## 总结

本章学习了：
- ✅ StorageContext 统一管理向量存储、文档存储和索引存储
- ✅ FAISS 作为向量存储后端的配置方式
- ✅ 持久化：`persist()` 保存 + `load_index_from_storage()` 加载
- ✅ 向已有索引插入新文档 `insert()`
- ✅ Embedding 模型配置和维度管理
- ✅ 对比 LangChain FAISS 的使用差异

## 下一章预告

**第五章：高级检索策略** —— 基础向量检索可能遗漏相关文档或引入噪音。下一章学习混合检索（向量 + BM25）、重排序、HyDE 查询变换等高级策略，这是 LlamaIndex 最核心的差异化能力。